# Week 6 - HAR: Dataset Understanding, Cleaning & Pre-processing

**Topics (record sections b, c)**
* b. Load the UCI HAR text files into pandas / NumPy
* c. Check data quality (missing, duplicate, infinite values) and **standardise** the features

## b. Dataset understanding
### Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             precision_score, recall_score, f1_score)

### Load the data
The `.txt` files are plain whitespace-separated tables, so we use `pd.read_csv(..., sep=r"\s+", header=None)` (`\s+` = one or more spaces).

* `X_*` -> pandas DataFrames (we attach the 561 feature names as column names)
* `y_*` -> NumPy integer arrays of activity ids
* `activity_names` -> a dict `{1: "WALKING", ...}`; we split each line only once (`maxsplit=1`) so names stay intact.

*(Heads-up: `features.txt` contains a few repeated feature names. They are made unique by appending the column index so pandas column selection is unambiguous.)*

In [2]:
# Local paths (the record used /content/... on Google Colab)
BASE = "data/har_raw/UCI HAR Dataset/"
X_train_path = BASE + "train/X_train.txt"
y_train_path = BASE + "train/y_train.txt"
X_test_path = BASE + "test/X_test.txt"
y_test_path = BASE + "test/y_test.txt"
features_path = BASE + "features.txt"
activities_path = BASE + "activity_labels.txt"

# Feature names (2nd column of features.txt)
features = pd.read_csv(features_path, sep=r"\s+", header=None)
feature_names = features.iloc[:, 1].tolist()

# Feature matrices: whitespace-separated numbers, no header
X_train = pd.read_csv(X_train_path, sep=r"\s+", header=None)
X_test = pd.read_csv(X_test_path, sep=r"\s+", header=None)

# The file has a few duplicated feature names (e.g. bandsEnergy), so we make them unique by suffixing the column index
seen, unique_names = {}, []
for i, n in enumerate(feature_names):
    unique_names.append(n if feature_names.count(n) == 1 else f"{n}_{i}")
feature_names = unique_names
X_train.columns = feature_names
X_test.columns = feature_names

# Labels
y_train = np.loadtxt(y_train_path, dtype=int)
y_test = np.loadtxt(y_test_path, dtype=int)

# Activity id -> name, keeping names that contain underscores/spaces
with open(activities_path, "r") as file:
    activity_names = {int(line.split(maxsplit=1)[0]): line.split(maxsplit=1)[1].strip() for line in file if line.strip()}

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)
print("Number of features:", len(feature_names))
print("Activity names:", activity_names)

Training data: (7352, 561)
Testing data: (2947, 561)
Number of features: 561
Activity names: {1: 'WALKING', 2: 'WALKING_UPSTAIRS', 3: 'WALKING_DOWNSTAIRS', 4: 'SITTING', 5: 'STANDING', 6: 'LAYING'}


In [3]:
X_train.iloc[:5, :8]

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y
0,0.288585,-0.020294,-0.132905,-0.995279,-0.983111,-0.913526,-0.995112,-0.983185
1,0.278419,-0.016411,-0.123520,-0.998245,-0.975300,-0.960322,-0.998807,-0.974914
2,0.279653,-0.019467,-0.113462,-0.995380,-0.967187,-0.978944,-0.996520,-0.963668
3,0.279174,-0.026201,-0.123283,-0.996091,-0.983403,-0.990675,-0.997099,-0.982750
4,0.276629,-0.016570,-0.115362,-0.998139,-0.980817,-0.990482,-0.998321,-0.979672


Each row is one 2.56 s window; each column one engineered feature. The values are already normalised to [-1, 1] by the dataset authors, but not to zero mean / unit variance - which many models prefer.

In [4]:
print(X_train.iloc[:, :6].describe().round(3))
print("\nTrain label counts:", dict(zip(*np.unique(y_train, return_counts=True))))

       tBodyAcc-mean()-X  tBodyAcc-mean()-Y  tBodyAcc-mean()-Z  \
count           7352.000           7352.000           7352.000   
mean               0.274             -0.018             -0.109   
std                0.070              0.041              0.057   
min               -1.000             -1.000             -1.000   
25%                0.263             -0.025             -0.121   
50%                0.277             -0.017             -0.109   
75%                0.288             -0.011             -0.098   
max                1.000              1.000              1.000   

       tBodyAcc-std()-X  tBodyAcc-std()-Y  tBodyAcc-std()-Z  
count          7352.000          7352.000          7352.000  
mean             -0.605            -0.511            -0.605  
std               0.449             0.503             0.419  
min              -1.000            -1.000            -1.000  
25%              -0.993            -0.978            -0.980  
50%              -0.946          

## c. Data cleaning and pre-processing
### Data quality checks
Always check before modelling:
* **Missing values** (`isnull`) - models can't handle NaNs,
* **Duplicate rows** - would leak between train and validation,
* **Infinite values** - break scalers and gradients.

In [5]:
print("Missing values in training data:", X_train.isnull().sum().sum())
print("Missing values in testing data:", X_test.isnull().sum().sum())
print("Duplicate training rows:", X_train.duplicated().sum())
print("Infinite values in training data:", np.isinf(X_train.to_numpy()).sum())

Missing values in training data: 0
Missing values in testing data: 0


Duplicate training rows: 0
Infinite values in training data: 0


### Standardise the features
`StandardScaler` rescales every feature to mean 0 and standard deviation 1: $z = (x-\mu)/\sigma$.

**Avoiding data leakage:** `fit_transform` on the *training* data learns $\mu,\sigma$; the *test* data only gets `transform` using those same numbers. Fitting on test data would let information about the test set leak into training.

### Encode labels
Ids 1-6 become 0-5 because Keras' `sparse_categorical_crossentropy` and array indexing expect classes starting at 0.

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)     # learn mean/std on TRAIN only
X_test_scaled = scaler.transform(X_test)           # reuse the train statistics on test
y_train_encoded = y_train - 1                      # ids 1-6 -> 0-5 (what Keras / sklearn metrics expect)
y_test_encoded = y_test - 1

print("Preprocessing completed.")
print("Training features:", X_train_scaled.shape)
print("Testing features:", X_test_scaled.shape)
print("Train mean (should be ~0):", X_train_scaled.mean().round(4), "| std (~1):", X_train_scaled.std().round(4))
print("Encoded labels:", np.unique(y_train_encoded))

Preprocessing completed.
Training features: (7352, 561)
Testing features: (2947, 561)
Train mean (should be ~0): 0.0 | std (~1): 1.0
Encoded labels: [0 1 2 3 4 5]


## Summary
* 7,352 train and 2,947 test windows x 561 features; no missing, duplicate or infinite values.
* Features standardised using train statistics only; labels shifted to 0-5.
* Week 7: exploratory data analysis. Week 8: modelling.